In [ ]:
#==========================================================================================================
#Notebook to perform cis-regulatory element (CRE) enrichment analysis.
#requires supplementary tables from Ruiz et al. 2019 available in the cre_enrichment folder.
#End-to-end CRE extraction + metadata + SNP annotation + enrichment.
#==========================================================================================================
import os
import re
import argparse
import pandas as pd
import numpy as np
from pyxlsb import open_workbook
from pybedtools import BedTool
from scipy.stats import mannwhitneyu, rankdata
import matplotlib.pyplot as plt


# ==============================================
#read data
# ==============================================

# paths
outdir='/home/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/Data/cre_enrichment/results'
xlsb = "/home/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/Data/cre_enrichment/Supplementary Tables_R1.xlsb" #supplementary tables from Ruiz's paper
gwas = "/home/harunnn/lstm_scratch/network_scratch/llineup/llineup-genomics/glm/allnets_results/gwas_snps.csv" #from merged snps gwas analysis
gff= "/home/harunnn/lstm_scratch/network_scratch/llineup/llineup-genomics/data/VectorBase-68_AgambiaePEST.gff" #downloaded from vectorbase

os.makedirs(outdir, exist_ok=True)


In [4]:
# ==============================================
#functions
# ==============================================

def read_xlsb_sheet(path, sheet):
    rows = []
    with open_workbook(path) as wb:
        with wb.get_sheet(sheet) as sh:
            for r in sh.rows():
                rows.append([c.v for c in r])
    df = pd.DataFrame(rows)

    # detect the header row (first row containing Literal 'Chromosome')
    header_row = df.index[df.apply(lambda r: (r == "Chromosome").any(), axis=1)][0]
    df.columns = df.iloc[header_row]
    df = df.iloc[header_row+1:].reset_index(drop=True)

    # clean column names
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("\u00A0"," ")    # NBSP
        .str.replace("\u200b","")     # zero-width
        .str.replace("’","'")
        .str.replace("–","-")
    )
    return df


def clean_coords(df, chr_col="Chromosome", start_col="Start", end_col="End"):
    #Convert and validate coordinates
    df = df.copy()
    df[chr_col]   = df[chr_col].astype(str).str.strip()
    df[start_col] = pd.to_numeric(df[start_col], errors="coerce")
    df[end_col]   = pd.to_numeric(df[end_col], errors="coerce")
    df = df.dropna(subset=[chr_col, start_col, end_col])
    df = df[df[end_col] > df[start_col]]
    df[start_col] = df[start_col].astype(int)
    df[end_col]   = df[end_col].astype(int)
    return df


In [5]:
# ==============================================        
# Load S4 sheet (CRE annotations)
# ==============================================
S4_SHEET = "S4_ATAC-seq THSs_b"

s4 = read_xlsb_sheet(xlsb, S4_SHEET)

# Coordinate cleaning
s4 = clean_coords(s4, "Chromosome", "Start", "End")

# Define target columns
target_cols = [
    "Chromosome",
    "Start",
    "End",
    "Annotation final",
    "Annotated Gene ID final",
    "Distance Peak Summit-Gene TSS/ATG (bp, HOMER)"
]

# Verify columns exist
for col in target_cols:
    if col not in s4.columns:
        raise ValueError(f"Missing expected S4 column: {col}")

# Select only the requested minimal subset
s4_min = s4[target_cols].copy()

# Rename coord columns for consistency (optional)
s4_min = s4_min.rename(columns={
    "Chromosome": "CHR",
    "Start": "start",
    "End": "end"
})

s4_min.head()


,CHR,start,end,Annotation final,Annotated Gene ID final,"Distance Peak Summit-Gene TSS/ATG (bp, HOMER)"
0,Mt,4,15338,Promoter,AGAP028372,24.0
1,Mt,4,15343,Promoter,AGAP028372,15.0
2,2R,4134187,4134416,Intronic,AGAP001389,5818.0
3,Mt,13068,13367,Promoter,AGAP028389,-575.0
4,Mt,13964,14499,Promoter,AGAP028391,-67.0


In [6]:
# ==============================================
# Write outputs
# ==============================================
s4_min[["CHR","start","end"]].to_csv(
    os.path.join(outdir,"S4_CRE_minimal.bed"),
    sep="\t", header=False, index=False
)

# Full metadata TSV
s4_min.to_csv(
    os.path.join(outdir,"S4_CRE_minimal.tsv"),
    sep="\t", index=False
)

print("Wrote:")
print(" -", os.path.join(outdir,"S4_CRE_minimal.bed"))
print(" -", os.path.join(outdir,"S4_CRE_minimal.tsv"))


Wrote:
 - /home/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/Data/cre_enrichment/results/S4_CRE_minimal.bed
 - /home/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/Data/cre_enrichment/results/S4_CRE_minimal.tsv


In [7]:
print("Rows:", len(s4_min))
print("\nAnnotation categories (Annotation final):")
print(s4_min["Annotation final"].value_counts())


Rows: 193005

Annotation categories (Annotation final):
Annotation final
Intronic              58896
Distal Intergenic     49432
Promoter              35323
Exonic                27408
5' UTR                13690
3' UTR                 2778
Downstream (<1kb)      2242
Downstream (1-2kb)     1620
Downstream (2-3kb)     1616
Name: count, dtype: int64


In [8]:
# ==============================================
# Prepare SNP BED file
# ==============================================

# Load your SNP file
# Update filename as needed — for example:
snp_df = pd.read_csv(gwas)

# Standardize CHR
snp_df["CHR"] = snp_df["CHR"].astype(str).str.strip()

# Signed score = sign(coef) * -log10(p)
snp_df["abs_score"] = -np.log10(snp_df["p"].astype(float))
snp_df["signed_score"] = np.sign(snp_df["coef"]) * snp_df["abs_score"]

# Prepare BED for intersection (0-based)
snp_bed = snp_df.assign(
    start=snp_df["BP"] - 1,
    end=snp_df["BP"]
)[["CHR","start","end"]]

snp_bed_file = "snp_positions.bed"
snp_bed.to_csv(os.path.join(outdir,snp_bed_file), sep="\t", header=False, index=False)

print("SNP BED written:", snp_bed_file)
snp_df.head()

SNP BED written: snp_positions.bed


,p,coef,snp.id,CHR,BP,p_adjusted,log10p,log10p_adj,abs_score,signed_score
0,0.24,0.4900,2R:24,2R,24,0.950634,0.619789,0.021987,0.619789,0.619789
1,0.35,0.1400,2R:114,2R,114,0.966639,0.455932,0.014736,0.455932,0.455932
2,0.31,-0.1600,2R:140,2R,140,0.962067,0.508638,0.016795,0.508638,-0.508638
3,0.96,0.0065,2R:217,2R,217,0.994388,0.017729,0.002444,0.017729,0.017729
4,0.13,-0.1900,2R:220,2R,220,0.911944,0.886057,0.040032,0.886057,-0.886057


In [9]:
snp_df

,p,coef,snp.id,CHR,BP,p_adjusted,log10p,log10p_adj,abs_score,signed_score
0,0.24,0.4900,2R:24,2R,24,0.950634,0.619789,0.021987,0.619789,0.619789
1,0.35,0.1400,2R:114,2R,114,0.966639,0.455932,0.014736,0.455932,0.455932
2,0.31,-0.1600,2R:140,2R,140,0.962067,0.508638,0.016795,0.508638,-0.508638
3,0.96,0.0065,2R:217,2R,217,0.994388,0.017729,0.002444,0.017729,0.017729
4,0.13,-0.1900,2R:220,2R,220,0.911944,0.886057,0.040032,0.886057,-0.886057
...,...,...,...,...,...,...,...,...,...,...
9898576,0.24,0.2500,3L:41659784,3L,41659784,0.950634,0.619789,0.021987,0.619789,0.619789
9898577,0.11,0.6900,3L:41659808,3L,41659808,0.897939,0.958607,0.046753,0.958607,0.958607
9898578,0.33,0.1600,3L:41659838,3L,41659838,0.964465,0.481486,0.015714,0.481486,0.481486
9898579,0.49,-0.3000,3L:41659908,3L,41659908,0.975714,0.309804,0.010677,0.309804,-0.309804


In [10]:
# ==============================================    
#Load datasets
# ==============================================
cre_df_path = os.path.join(outdir,"S4_CRE_minimal.tsv")
cre_df = pd.read_csv(cre_df_path, sep="\t")

cre_bed = cre_df[["CHR","start","end"]]
cre_bed_file = "S4_CRE_minimal.bed"

cre_bed.to_csv(cre_bed_file, sep="\t", header=False, index=False)
print("CRE BED ready:", cre_bed_file)

cre_df.head()

CRE BED ready: S4_CRE_minimal.bed


,CHR,start,end,Annotation final,Annotated Gene ID final,"Distance Peak Summit-Gene TSS/ATG (bp, HOMER)"
0,Mt,4,15338,Promoter,AGAP028372,24.0
1,Mt,4,15343,Promoter,AGAP028372,15.0
2,2R,4134187,4134416,Intronic,AGAP001389,5818.0
3,Mt,13068,13367,Promoter,AGAP028389,-575.0
4,Mt,13964,14499,Promoter,AGAP028391,-67.0


In [11]:
cre_bed

,CHR,start,end
0,Mt,4,15338
1,Mt,4,15343
2,2R,4134187,4134416
3,Mt,13068,13367
4,Mt,13964,14499
...,...,...,...
193000,2L,7644624,7644672
193001,2L,7695151,7695211
193002,2L,45465713,45465781
193003,2L,45507259,45507306


In [12]:
# ==============================================
# Intersect SNPs with CREs
# ==============================================

# Explicit paths
snp_bed_file = os.path.abspath("snp_positions.bed")
cre_bed_file = os.path.abspath("S4_CRE_minimal.bed")

# Safety check
print("Reading:")
print(" - SNP BED:", snp_bed_file)
print(" - CRE BED:", cre_bed_file)

if not os.path.exists(snp_bed_file):
    raise FileNotFoundError(f"SNP BED not found: {snp_bed_file}")

if not os.path.exists(cre_bed_file):
    raise FileNotFoundError(f"CRE BED not found: {cre_bed_file}")

# Load BED as BedTools
bt_snp = BedTool(snp_bed_file)
bt_cre = BedTool(cre_bed_file)

# Perform intersection
overlaps = bt_snp.intersect(bt_cre, wa=True, wb=True)

print("Loaded BED files and found overlaps:", len(overlaps))


records = []
for o in overlaps:
    snp_chr   = o[0]
    snp_start = int(o[1])
    snp_end   = int(o[2])

    cre_chr   = o[3]
    cre_start = int(o[4])
    cre_end   = int(o[5])

    records.append((snp_chr, snp_start, snp_end, cre_chr, cre_start, cre_end))

ov_df = pd.DataFrame(records, columns=[
    "CHR","snp_start","snp_end","cre_CHR","cre_start","cre_end"
])

print("Total SNPs overlapping CREs:", len(ov_df))
ov_df.head()


Reading:
 - SNP BED: /mnt/user_shares/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/scripts_notebooks/genomewide_association/cre_enrichment/snp_positions.bed
 - CRE BED: /mnt/user_shares/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/scripts_notebooks/genomewide_association/cre_enrichment/S4_CRE_minimal.bed
Loaded BED files and found overlaps: 1420245
Total SNPs overlapping CREs: 1420245


,CHR,snp_start,snp_end,cre_CHR,cre_start,cre_end
0,2R,1235,1236,2R,1215,1273
1,2R,1238,1239,2R,1215,1273
2,2R,1268,1269,2R,1215,1273
3,2R,20217,20218,2R,20183,20229
4,2R,27223,27224,2R,27177,27244


In [13]:
overlaps 

<BedTool(/tmp/pybedtools.e4lg8xy0.tmp)>

In [14]:
#==============================================
# Merge SNP metadata
#==============================================
ov_df = ov_df.merge(
    snp_df,
    left_on=["CHR","snp_end"],
    right_on=["CHR","BP"],
    how="left"
)

# Merge CRE metadata
ov_df = ov_df.merge(
    cre_df,
    left_on=["cre_CHR","cre_start","cre_end"],
    right_on=["CHR","start","end"],
    how="left",
    suffixes=("","_cre")
)

print("SNP–CRE merged table:")
ov_df.head()


SNP–CRE merged table:


,CHR,snp_start,snp_end,cre_CHR,cre_start,cre_end,p,coef,snp.id,BP,...,log10p,log10p_adj,abs_score,signed_score,CHR_cre,start,end,Annotation final,Annotated Gene ID final,"Distance Peak Summit-Gene TSS/ATG (bp, HOMER)"
0,2R,1235,1236,2R,1215,1273,0.016,-1.00,2R:1236,1236,...,1.795880,0.093988,1.795880,-1.795880,2R,1215,1273,Distal Intergenic,AGAP001096,-2189.0
1,2R,1238,1239,2R,1215,1273,0.410,-0.21,2R:1239,1239,...,0.387216,0.012393,0.387216,-0.387216,2R,1215,1273,Distal Intergenic,AGAP001096,-2189.0
2,2R,1268,1269,2R,1215,1273,0.110,-0.27,2R:1269,1269,...,0.958607,0.046753,0.958607,-0.958607,2R,1215,1273,Distal Intergenic,AGAP001096,-2189.0
3,2R,20217,20218,2R,20183,20229,0.410,0.33,2R:20218,20218,...,0.387216,0.012393,0.387216,0.387216,2R,20183,20229,5' UTR,AGAP001098,408.0
4,2R,27223,27224,2R,27177,27244,0.550,-0.21,2R:27224,27224,...,0.259637,0.009770,0.259637,-0.259637,2R,27177,27244,Intronic,AGAP001099,4213.0


In [15]:
#==============================================
# Write final SNP→CRE mapping
#==============================================

final = ov_df[[
    "snp.id","CHR", "BP", "p", "coef","cre_start","cre_end",
    "abs_score", "signed_score",
    "Annotation final",
    "Annotated Gene ID final",
    "Distance Peak Summit-Gene TSS/ATG (bp, HOMER)"
]].rename(columns={
    #"snp.id": "snp_id",
    "Annotation final": "CRE_annotation",
    "Annotated Gene ID final": "Gene_ID",
    "Distance Peak Summit-Gene TSS/ATG (bp, HOMER)": "Distance_to_gene"
})
final.to_csv(
    os.path.join(outdir,"SNP_CRE_mapping.tsv"),
    sep="\t", index=False
)


print("Final SNP→CRE mapping written to SNP_CRE_mapping.tsv")
final


Final SNP→CRE mapping written to SNP_CRE_mapping.tsv


,snp.id,CHR,BP,p,coef,cre_start,cre_end,abs_score,signed_score,CRE_annotation,Gene_ID,Distance_to_gene
0,2R:1236,2R,1236,0.016,-1.00,1215,1273,1.795880,-1.795880,Distal Intergenic,AGAP001096,-2189.0
1,2R:1239,2R,1239,0.410,-0.21,1215,1273,0.387216,-0.387216,Distal Intergenic,AGAP001096,-2189.0
2,2R:1269,2R,1269,0.110,-0.27,1215,1273,0.958607,-0.958607,Distal Intergenic,AGAP001096,-2189.0
3,2R:20218,2R,20218,0.410,0.33,20183,20229,0.387216,0.387216,5' UTR,AGAP001098,408.0
4,2R:27224,2R,27224,0.550,-0.21,27177,27244,0.259637,-0.259637,Intronic,AGAP001099,4213.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1427896,3L:41659784,3L,41659784,0.240,0.25,41659703,41659797,0.619789,0.619789,Distal Intergenic,AGAP012389,33343.0
1427897,3L:41659908,3L,41659908,0.490,-0.30,41659906,41660346,0.309804,-0.309804,Distal Intergenic,AGAP012390,-33439.0
1427898,3L:41659908,3L,41659908,0.490,-0.30,41659901,41660354,0.309804,-0.309804,Distal Intergenic,AGAP012390,-33381.0
1427899,3L:41659967,3L,41659967,0.180,-0.34,41659906,41660346,0.744727,-0.744727,Distal Intergenic,AGAP012390,-33439.0


In [16]:
print("Total SNPs:", len(snp_df))
print("SNPs overlapping CREs:", len(final))
print("\nCRE categories hit:")
print(final["CRE_annotation"].value_counts())


Total SNPs: 9898581
SNPs overlapping CREs: 1427901

CRE categories hit:
CRE_annotation
Intronic              487047
Promoter              337262
Distal Intergenic     330165
5' UTR                108689
Exonic                108359
Downstream (<1kb)      18285
3' UTR                 14749
Downstream (2-3kb)     12018
Downstream (1-2kb)     11327
Name: count, dtype: int64


In [17]:
# Count how many CRE regions each SNP overlaps and drop duplicates

final_dedup = final.drop_duplicates(
    subset=["CHR", "BP", "CRE_annotation"],
    keep="first"
)
print("Overlap rows before:", len(final))
print("After dropping same-annotation duplicates:", len(final_dedup))


Overlap rows before: 1427901
After dropping same-annotation duplicates: 718361


In [18]:
# ==============================================
# Collapse to one CRE per SNP   
# ==============================================

HIERARCHY = [
    "Promoter",
    "5' UTR",
    "3' UTR",
    "Exonic",
    "Intronic",
    "Downstream (<1kb)",
    "Downstream (1-2kb)",
    "Downstream (2-3kb)",
    "Distal Intergenic",
]
rank_map = {cre:i for i,cre in enumerate(HIERARCHY,1)}

dedup = final_dedup.copy()
dedup["rank"] = dedup["CRE_annotation"].map(rank_map)

dedup["abs_dist"] = dedup["Distance_to_gene"].abs()
dedup["cre_width"] = dedup["cre_end"] - dedup["cre_start"]
dedup["snp_key"] = dedup["CHR"].astype(str) + ":" + dedup["BP"].astype(str)

final_collapsed = (
    dedup.sort_values(
        ["snp_key","rank","abs_dist","cre_width"],
        ascending=[True,True,True,True]
    )
    .drop_duplicates("snp_key")
)

print("Original overlap rows:", len(final))
print("Collapsed unique SNP rows:", len(final_collapsed))
print("Unique SNPs:", final_collapsed["snp_key"].nunique())



Original overlap rows: 1427901
Collapsed unique SNP rows: 685035
Unique SNPs: 685035


In [19]:
final_collapsed

,snp.id,CHR,BP,p,coef,cre_start,cre_end,abs_score,signed_score,CRE_annotation,Gene_ID,Distance_to_gene,rank,abs_dist,cre_width,snp_key
378041,2L:10006512,2L,10006512,0.280,-0.44,10006471,10006545,0.552842,-0.552842,Intronic,AGAP005096,17817.0,5,17817.0,74,2L:10006512
378042,2L:10006523,2L,10006523,0.580,0.13,10006471,10006545,0.236572,0.236572,Intronic,AGAP005096,17817.0,5,17817.0,74,2L:10006523
378043,2L:10012322,2L,10012322,0.250,-0.14,10012319,10012402,0.602060,-0.602060,Exonic,AGAP005096,11980.0,4,11980.0,83,2L:10012322
378044,2L:10012328,2L,10012328,0.360,-0.32,10012319,10012402,0.443697,-0.443697,Exonic,AGAP005096,11980.0,4,11980.0,83,2L:10012328
378045,2L:10012349,2L,10012349,0.059,-0.39,10012319,10012402,1.229148,-1.229148,Exonic,AGAP005096,11980.0,4,11980.0,83,2L:10012349
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145994,X:9998221,X,9998221,0.160,0.58,9998078,9998311,0.795880,0.795880,Intronic,AGAP000562,12881.0,5,12881.0,233,X:9998221
1145998,X:9998231,X,9998231,0.068,0.68,9998078,9998311,1.167491,1.167491,Intronic,AGAP000562,12881.0,5,12881.0,233,X:9998231
1146002,X:9998235,X,9998235,0.220,0.47,9998078,9998311,0.657577,0.657577,Intronic,AGAP000562,12881.0,5,12881.0,233,X:9998235
1146006,X:9998261,X,9998261,0.220,0.22,9998078,9998311,0.657577,0.657577,Intronic,AGAP000562,12881.0,5,12881.0,233,X:9998261


In [20]:
snp_df

,p,coef,snp.id,CHR,BP,p_adjusted,log10p,log10p_adj,abs_score,signed_score
0,0.24,0.4900,2R:24,2R,24,0.950634,0.619789,0.021987,0.619789,0.619789
1,0.35,0.1400,2R:114,2R,114,0.966639,0.455932,0.014736,0.455932,0.455932
2,0.31,-0.1600,2R:140,2R,140,0.962067,0.508638,0.016795,0.508638,-0.508638
3,0.96,0.0065,2R:217,2R,217,0.994388,0.017729,0.002444,0.017729,0.017729
4,0.13,-0.1900,2R:220,2R,220,0.911944,0.886057,0.040032,0.886057,-0.886057
...,...,...,...,...,...,...,...,...,...,...
9898576,0.24,0.2500,3L:41659784,3L,41659784,0.950634,0.619789,0.021987,0.619789,0.619789
9898577,0.11,0.6900,3L:41659808,3L,41659808,0.897939,0.958607,0.046753,0.958607,0.958607
9898578,0.33,0.1600,3L:41659838,3L,41659838,0.964465,0.481486,0.015714,0.481486,0.481486
9898579,0.49,-0.3000,3L:41659908,3L,41659908,0.975714,0.309804,0.010677,0.309804,-0.309804


In [21]:
# ==============================================
# Merge ALL SNPs with CRE assignments
# ==============================================

# Merge ALL SNPs with CRE assignments
all_snp = snp_df.merge(
    final_collapsed[["snp.id", "CRE_annotation", "Gene_ID"]],
    on=["snp.id"],
    how="left"
)

# Fill unmapped SNPs with "non_THS"
all_snp["CRE_annotation"] = all_snp["CRE_annotation"].fillna("non_THS")

# Keep required columns
all_snp = all_snp[[
    "CHR","BP","p","coef",
    "abs_score","signed_score",
    "CRE_annotation","Gene_ID"
]]

print("Total SNPs in GWAS:", len(snp_df))
print("Total SNPs in final all_snp:", len(all_snp))
print("CRE annotation frequencies:")
print(all_snp["CRE_annotation"].value_counts())


Total SNPs in GWAS: 9898581
Total SNPs in final all_snp: 9898581
CRE annotation frequencies:
CRE_annotation
non_THS               9213546
Intronic               230386
Distal Intergenic      165936
Promoter               140740
Exonic                  81828
5' UTR                  37935
3' UTR                   9176
Downstream (<1kb)        8203
Downstream (2-3kb)       5580
Downstream (1-2kb)       5251
Name: count, dtype: int64


In [22]:
#==============================================
# Rank based enrichment
#==============================================
df_rank = all_snp[["CRE_annotation","abs_score"]].dropna().copy()
df_rank["rank"] = rankdata(df_rank["abs_score"].values, method="average")  # ascending ranks

N = len(df_rank)
results_rank = []

for cre in sorted(df_rank["CRE_annotation"].unique()):
    mask = (df_rank["CRE_annotation"] == cre)
    n1 = mask.sum()
    n2 = N - n1
    if n1 == 0 or n2 == 0:
        continue

    # Rank-sum statistic
    R1 = df_rank.loc[mask, "rank"].sum()
    U1 = R1 - n1*(n1+1)/2.0

    # Effect size (AUC)
    auc = U1 / (n1*n2)

    # p-value from Mann–Whitney test
    pval = mannwhitneyu(
        df_rank.loc[mask, "abs_score"],
        df_rank.loc[~mask, "abs_score"],
        alternative="two-sided"
    ).pvalue

    results_rank.append((cre, n1, n2, auc, pval))

rank_df = pd.DataFrame(
    results_rank,
    columns=["CRE","n_in_group","n_out_group","AUC_rank_enrichment","pvalue"]
).sort_values("AUC_rank_enrichment", ascending=False)

rank_df_path = os.path.join(outdir, "rank_based_enrichment_auc.tsv")
rank_df.to_csv(rank_df_path, sep="\t", index=False)

print("Wrote rank-based AUC table:", rank_df_path)


# not significant enrichments have AUC around 0.5

Wrote rank-based AUC table: /home/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/Data/cre_enrichment/results/rank_based_enrichment_auc.tsv


In [ ]:
# ------------------------------------------------------------
# 4) Permutation enrichment on top SNPs (p < 0.001) by shuffling annotations
# ------------------------------------------------------------
from collections import Counter, defaultdict
from tqdm import trange
alpha = 0.001
n_perm = 1000
top_mask = all_snp["p"] < alpha
top = all_snp[top_mask].copy()
bg  = all_snp.copy()

print(f"Top SNPs (p<{alpha}): {len(top):,}")

# Observed counts per CRE among top
obs_counts = top["CRE_annotation"].value_counts().to_dict()
obs_any_cre = (top["CRE_annotation"] != "non_THS").sum()

# Build label vector to shuffle (same size as BG)
labels = bg["CRE_annotation"].values.copy()

rng = np.random.default_rng(42)

def perm_once():
    # shuffle labels across ALL SNPs (keeps label frequencies but breaks association with scores)
    shuffled = labels.copy()
    rng.shuffle(shuffled)
    # take the top set by the same mask size (same number of rows)
    # We preserve the same top set indices (by position), which is simplest and valid.
    top_labels = shuffled[top_mask.values]
    # Count any CRE vs non_THS
    any_cre = np.sum(top_labels != "non_THS")
    # Also per-category counts (optional)
    uniq, cnts = np.unique(top_labels, return_counts=True)
    return any_cre, dict(zip(uniq, cnts))

perm_any = []
perm_by_cat = defaultdict(list)

for _ in trange(n_perm, desc="Permuting"):
    any_cre, cat_counts = perm_once()
    perm_any.append(any_cre)
    for k,v in cat_counts.items():
        perm_by_cat[k].append(v)

perm_any = np.array(perm_any)

# Empirical p-value for "any CRE" among top SNPs
obs_any = obs_any_cre
p_emp_any = (np.sum(perm_any >= obs_any) + 1) / (n_perm + 1)

perm_summary = {
    "metric": ["any_CRE_in_top"],
    "observed": [int(obs_any)],
    "mean_null": [float(np.mean(perm_any))],
    "sd_null": [float(np.std(perm_any, ddof=1))],
    "empirical_p": [float(p_emp_any)]
}
perm_summary_df = pd.DataFrame(perm_summary)
perm_summary_path = os.path.join(outdir, "permutation_enrichment_anyCRE.tsv")
perm_summary_df.to_csv(perm_summary_path, sep="\t", index=False)
print("Wrote:", perm_summary_path)

# Per-category empirical p-values (optional; many tests → consider FDR later)
rows = []
for cre, obs_ct in obs_counts.items():
    dist = np.array(perm_by_cat.get(cre, [0]*n_perm))
    p_emp = (np.sum(dist >= obs_ct) + 1) / (n_perm + 1)
    rows.append((cre, int(obs_ct), float(dist.mean()), float(dist.std(ddof=1)), float(p_emp)))

perm_by_cat_df = pd.DataFrame(rows, columns=["CRE","observed","mean_null","sd_null","empirical_p"]) \
                 .sort_values("empirical_p")
perm_by_cat_path = os.path.join(outdir, "permutation_enrichment_by_CRE.tsv")
perm_by_cat_df.to_csv(perm_by_cat_path, sep="\t", index=False)
print("Wrote:", perm_by_cat_path)

# ------------------------------------------------------------
# 5) Top CRE-linked genes (export lists for downstream GSEA/ORA)
# ------------------------------------------------------------
# Define top set again (same alpha)
all_cre = all_snp[all_snp["CRE_annotation"] != "non_THS"].copy()

# Keep nearest gene (Gene_ID may be blank for some; dropna)
gene_list = (all_cre["Gene_ID"].astype(str).str.strip())
gene_list = gene_list[gene_list != ""]
gene_list = gene_list[gene_list != "nan"]

# Unique genes + ranked by best SNP score seen per gene
best_by_gene = (all_cre.assign(best_abs=all_cre.groupby("Gene_ID")["abs_score"].transform("max"))
                        .drop_duplicates("Gene_ID")
                        .sort_values("best_abs", ascending=False)[["Gene_ID","best_abs"]])

gene_list_path = os.path.join(outdir, "all_cre_linked_genes_unique.txt")
best_by_gene_path = os.path.join(outdir, "all_cre_linked_genes_ranked.tsv")

gene_list.drop_duplicates().to_csv(gene_list_path, index=False, header=False)
best_by_gene.to_csv(best_by_gene_path, sep="\t", index=False)

print("Wrote:")
print(" -", gene_list_path, "(unique IDs; paste into your favorite GSEA/ORA)")
print(" -", best_by_gene_path, "(ranked by best SNP abs_score)")

# ------------------------------------------------------------
# 6) Save compact summaries
# ------------------------------------------------------------
rank_df.to_csv(os.path.join(outdir, "summary_rank_auc.tsv"), sep="\t", index=False)
perm_summary_df.to_csv(os.path.join(outdir, "summary_perm_anyCRE.tsv"), sep="\t", index=False)
perm_by_cat_df.to_csv(os.path.join(outdir, "summary_perm_by_cat.tsv"), sep="\t", index=False)



Top SNPs (p<0.001): 14,454


Permuting: 100%|██████████| 1000/1000 [09:34<00:00,  1.74it/s]

Wrote: /home/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/Data/cre_enrichment/results/permutation_enrichment_anyCRE.tsv
Wrote: /home/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/Data/cre_enrichment/results/permutation_enrichment_by_CRE.tsv
Wrote:
 - /home/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/Data/cre_enrichment/results/top_CRE_linked_genes_unique.txt (unique IDs; paste into your favorite GSEA/ORA)
 - /home/harunnn/lstm_scratch/network_scratch/llineup/llineup_publication/Data/cre_enrichment/results/top_CRE_linked_genes_ranked.tsv (ranked by best SNP abs_score)
